# IC checks - 1h candles

Archival first-pass research notebook for Easley-style crypto market microstructure features using **1-hour Binance spot candles**.

Current interpretation: these studies did not establish microstructure as a primary alpha. Keep this notebook for execution-quality and liquidity-context diagnostics only. The current deployment-style research uses `data/features/live_1h_features_30m.csv` and compares fixed V1 versus scheduled-refit V2 over the same OOS window.

Important: the Easley paper computes 50/100-bar measures on one-minute bars. In this notebook, `window=50` and `window=100` mean **50/100 one-hour bars**.


## What the IC tables mean

**Forward return target**: `forward_return_HORIZON` is the percentage return over the next `HORIZON` one-hour bars, computed as `close[t+HORIZON] / close[t] - 1` within each pair.

**Pearson IC**: ordinary linear correlation between a feature and the forward return.

**Spearman IC**: Pearson correlation after ranking the feature and target. This is the more robust first-pass check when we care about monotonic ranking rather than linear scale.

**Pooled IC**: one IC per feature/window/method after stacking every valid `(pair, time)` observation together. It is useful as a broad screen, but it treats cross-asset and overlapping-window observations as one large sample, so do not read it as a clean significance test.

**Per-pair IC**: the same IC computed separately for each symbol. This helps detect whether a pooled result is broad-based or driven by a small number of assets.


In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# Jupyter often runs this notebook with cwd=notebooks/. Add the repo root so
# imports from bot.* work whether the notebook is opened from root or notebooks.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = next(
    parent for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (parent / "bot").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bot.binance_data import BinanceData  # noqa: E402
from bot.config import BINANCE_SYMBOL_MAP, TRADEABLE_COINS  # noqa: E402
from bot.forward_ic import add_forward_return, compute_univariate_ics  # noqa: E402
from bot.microstructure import MICROSTRUCTURE_COLUMNS, compute_microstructure_measures  # noqa: E402

INTERVAL = "1h"
LIMIT = 1000
WINDOWS = (50, 100)
HORIZON = 24
MIN_PERIODS = 30
OUTPUT_DIR = NOTEBOOK_DIR
BASE_PREFIX = f"ic_checks_{INTERVAL}_candles"
RESULT_PREFIX = f"{BASE_PREFIX}_h{HORIZON}"

RAW_PATH = OUTPUT_DIR / f"{BASE_PREFIX}_raw.csv"
FEATURES_PATH = OUTPUT_DIR / f"{RESULT_PREFIX}_features.csv"
POOLED_PATH = OUTPUT_DIR / f"{RESULT_PREFIX}_pooled.csv"
PER_PAIR_PATH = OUTPUT_DIR / f"{RESULT_PREFIX}_per_pair.csv"
METADATA_PATH = OUTPUT_DIR / f"{RESULT_PREFIX}_metadata.json"

# Keep False to avoid live network fetch. With False, the notebook reuses the
# cached raw 1h candle file and recomputes features/ICs for the chosen HORIZON.
# Set True only when you want fresh live Binance candles.
RUN_LIVE_FETCH = False


In [ ]:
def usd_spot_pairs() -> list[str]:
    return [pair for pair in TRADEABLE_COINS if pair.endswith("/USD") and pair in BINANCE_SYMBOL_MAP]


def fetch_universe(pairs: list[str], interval: str, limit: int, sleep_seconds: float = 0.05):
    feed = BinanceData()
    frames = []
    loaded = []
    failed = []
    for pair in pairs:
        candles = feed.fetch_klines(pair, interval=interval, limit=limit)
        if not candles:
            failed.append(pair)
            continue
        frame = pd.DataFrame(candles)
        frame["pair"] = pair
        frame["binance_symbol"] = BINANCE_SYMBOL_MAP[pair]
        frames.append(frame)
        loaded.append(pair)
        time.sleep(sleep_seconds)
    if not frames:
        raise RuntimeError("No Binance candles were loaded.")
    return pd.concat(frames, ignore_index=True), loaded, failed


def build_feature_frame(candles: pd.DataFrame, windows: tuple[int, ...], horizon: int):
    frames = []
    for window in windows:
        for _, group in candles.groupby("pair", sort=False):
            group = group.sort_values("open_time").reset_index(drop=True)
            features = compute_microstructure_measures(group, window=window)
            features = add_forward_return(
                features,
                horizon=horizon,
                price_col="close",
                target_col=f"forward_return_{horizon}",
            )
            features["microstructure_window"] = window
            frames.append(features)
    return pd.concat(frames, ignore_index=True)


def compute_ic_tables(features: pd.DataFrame, horizon: int):
    target_col = f"forward_return_{horizon}"
    pooled_frames = []
    per_pair_frames = []
    for window, group in features.groupby("microstructure_window", sort=True):
        pooled = compute_univariate_ics(
            group,
            feature_cols=MICROSTRUCTURE_COLUMNS,
            target_col=target_col,
            methods=("pearson", "spearman"),
            horizon=horizon,
            min_periods=MIN_PERIODS,
        )
        pooled["microstructure_window"] = int(window)
        pooled_frames.append(pooled)

        per_pair = compute_univariate_ics(
            group,
            feature_cols=MICROSTRUCTURE_COLUMNS,
            target_col=target_col,
            methods=("pearson", "spearman"),
            horizon=horizon,
            min_periods=MIN_PERIODS,
            group_col="pair",
        )
        per_pair["microstructure_window"] = int(window)
        per_pair_frames.append(per_pair)
    return pd.concat(pooled_frames, ignore_index=True), pd.concat(per_pair_frames, ignore_index=True)


def run_or_load():
    pairs = usd_spot_pairs()
    if RUN_LIVE_FETCH or not RAW_PATH.exists():
        candles, loaded, failed = fetch_universe(pairs, INTERVAL, LIMIT)
        candles.to_csv(RAW_PATH, index=False)
    else:
        candles = pd.read_csv(RAW_PATH)
        loaded = sorted(candles["pair"].dropna().unique().tolist())
        failed = [pair for pair in pairs if pair not in set(loaded)]

    # Always recompute features and ICs from raw candles for the selected HORIZON.
    # This avoids accidentally reading horizon-1 cached ICs after changing HORIZON.
    features = build_feature_frame(candles, WINDOWS, HORIZON)
    pooled, per_pair = compute_ic_tables(features, HORIZON)
    metadata = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "interval": INTERVAL,
        "limit": LIMIT,
        "horizon": HORIZON,
        "windows": list(WINDOWS),
        "requested_pairs": len(pairs),
        "loaded_pairs": loaded,
        "failed_pairs": failed,
        "candles_rows": int(len(candles)),
        "feature_rows": int(len(features)),
        "raw_source": str(RAW_PATH),
        "live_fetch": RUN_LIVE_FETCH,
    }
    features.to_csv(FEATURES_PATH, index=False)
    pooled.to_csv(POOLED_PATH, index=False)
    per_pair.to_csv(PER_PAIR_PATH, index=False)
    METADATA_PATH.write_text(json.dumps(metadata, indent=2, sort_keys=True))
    return candles, features, pooled, per_pair, metadata


candles, features, pooled, per_pair, metadata = run_or_load()
metadata


In [ ]:
from IPython.display import HTML, display


def svg_bar_chart(df, value_col, label_col, title, width=920, row_h=28, left=260, right=90):
    rows = df[[label_col, value_col]].dropna().sort_values(value_col).copy()
    height = 50 + row_h * len(rows)
    plot_w = width - left - right
    max_abs = max(float(rows[value_col].abs().max()), 0.001)
    zero_x = left + plot_w / 2
    scale = (plot_w / 2) / max_abs
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    parts.append('<rect width="100%" height="100%" fill="#ffffff"/>')
    parts.append(f'<text x="12" y="24" font-family="Arial" font-size="16" font-weight="700">{title}</text>')
    parts.append(f'<line x1="{zero_x:.1f}" y1="38" x2="{zero_x:.1f}" y2="{height-12}" stroke="#555" stroke-width="1"/>')
    for i, row in enumerate(rows.itertuples(index=False)):
        label = str(getattr(row, label_col)).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        value = float(getattr(row, value_col))
        y = 48 + i * row_h
        bar_w = abs(value) * scale
        x = zero_x if value >= 0 else zero_x - bar_w
        color = "#2f7d32" if value >= 0 else "#b23a3a"
        parts.append(f'<text x="12" y="{y+14}" font-family="Arial" font-size="12">{label}</text>')
        parts.append(f'<rect x="{x:.1f}" y="{y}" width="{bar_w:.1f}" height="18" fill="{color}" opacity="0.82"/>')
        tx = zero_x + bar_w + 6 if value >= 0 else zero_x - bar_w - 58
        parts.append(f'<text x="{tx:.1f}" y="{y+14}" font-family="Arial" font-size="12">{value:+.4f}</text>')
    parts.append("</svg>")
    return "".join(parts)


## Pooled IC summary

Each row is one microstructure feature, one IC method, and one microstructure lookback window. The `n` column is the number of valid feature/target observations after NaNs are removed.

In [ ]:
pooled.sort_values(["microstructure_window", "method", "feature"])

## Pooled IC plot

Negative values mean higher feature values tend to precede lower forward returns at the configured `HORIZON`. The cell below regenerates the plot from the current `pooled` table each time you rerun the notebook.


In [ ]:
pooled_plot = pooled.copy()
pooled_plot["label"] = pooled_plot.apply(
    lambda r: f"{r['feature']} w{int(r['microstructure_window'])} {r['method']}", axis=1
)
display(HTML(svg_bar_chart(pooled_plot, "ic", "label", f"Pooled IC vs forward {HORIZON}h return")))


## Largest absolute pooled IC rows

This is a quick ranking of the strongest pooled relationships. It is not a multiple-testing-adjusted result and should not be treated as a standalone trading signal.

In [ ]:
pooled.assign(abs_ic=pooled["ic"].abs()).sort_values("abs_ic", ascending=False).head(10)

## Per-pair IC summary

This table summarizes the distribution of per-pair ICs. A feature with a small pooled IC but a consistent per-pair mean/median may still be useful. A feature with a few large per-pair ICs and weak mean/median is more likely pair-specific or noisy.

In [ ]:
per_pair_summary = (
    per_pair.groupby(["microstructure_window", "feature", "method"])["ic"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
    .sort_values(["microstructure_window", "method", "mean"])
)
per_pair_summary


## Mean per-pair Spearman IC plot

This averages each pair's Spearman IC by feature/window. It gives a better sense of broad-based direction than the pooled table alone. The cell below regenerates from the current `per_pair` table.


In [ ]:
per_pair_plot = (
    per_pair[per_pair["method"].eq("spearman")]
    .groupby(["microstructure_window", "feature"])["ic"]
    .agg(mean="mean", median="median")
    .reset_index()
)
per_pair_plot["label"] = per_pair_plot.apply(
    lambda r: f"{r['feature']} w{int(r['microstructure_window'])}", axis=1
)
display(HTML(svg_bar_chart(
    per_pair_plot,
    "mean",
    "label",
    f"Mean per-pair Spearman IC vs forward {HORIZON}h return",
    width=880,
    left=215,
)))


## Strongest individual pair ICs

These are useful for diagnostics, but they are also where data mining risk is highest. Treat them as candidates for further out-of-sample checks, not as conclusions.

In [ ]:
per_pair.assign(abs_ic=per_pair["ic"].abs()).sort_values("abs_ic", ascending=False).head(15)

## Current 24h horizon read

For `HORIZON = 24`, the pooled ICs are weaker than the 6h test but still mostly negative for VPIN and Roll. VPIN remains the strongest pooled Spearman feature at `window=50` (`IC ~= -0.028`), with Roll close behind (`window=100 Spearman IC ~= -0.024`, `window=50 Spearman IC ~= -0.021`).

The result suggests that toxicity/illiquidity features still lean bearish over the next day, but the effect is less concentrated and less compelling than at the 6h horizon. This makes 6h look like the more natural forward-return horizon for these 1h-bar microstructure features in the current sample.
